# Stage B v5.2 — multi-sample Pass-1 (union of 3 landscapes)

**v5.1 result**: K-cap F1=0.333 on val_009 (was 0.000 in v4, 0.038 in v5). Caught Art. 285, 286×2, 100 BGG, 292 ZGB.
**v5.1 problem**: Pass-1 played too safe under anti-hallucination guidance — produced a narrow landscape missing Art. 277 ZGB, 291 ZGB, 308 ZGB (real Swiss provisions). Pass-2 then defaulted to Rule D for those.

**v5.2 fix**: run Pass-1 three times with seeds 42/43/44. Union the article lists per (aspect, category). The union captures provisions any single run might miss while keeping the anti-hallucination filter (article ranges are validated on the merged set).

**Cost**: Pass-1 ~4 min (3× a single run), Pass-2 unchanged ~6 min. Total ~10 min.

## Phase 0 — Setup

In [ ]:
import os, sys, subprocess, json, time, gc, io, re
from pathlib import Path
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

IS_COLAB = "google.colab" in sys.modules
print(f"Colab: {IS_COLAB}")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([
        "pip", "install", "-q", "-U",
        "vllm>=0.9.1", "transformers>=4.51.0", "pandas==2.2.3",
        "pyarrow==16.1.0", "numpy==1.26.4", "tqdm",
    ], check=True)


## Phase 1 — Paths and config

In [ ]:
DRIVE_ROOT  = Path("/content/drive/MyDrive/swiss_law")
STAGE_B_IN  = DRIVE_ROOT / "research" / "stage_b_input" / "stage_b_input.parquet"
VAL_CSV     = DRIVE_ROOT / "data"     / "val.csv"
ALL_TARGETS = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "snapshot" / "all_targets.json"
GOLD_SETS   = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "snapshot" / "gold_doc_sets.json"
VAL_ASPECTS = DRIVE_ROOT / "research" / "concept_embedding_path" / "multi_aspect" / "val_aspects.parquet"

OUT_DIR = DRIVE_ROOT / "research" / "stage_b_grounded_llm" / "v52_single_query"
OUT_DIR.mkdir(parents=True, exist_ok=True)

QID                  = "val_009"
TOP_K                = 2000
LLM_MODEL            = "Qwen/Qwen3-32B-AWQ"
PASS1_MAX_TOKENS     = 4096
PASS1_MAX_MODEL_LEN  = 8192
PASS2_MAX_TOKENS     = 1024
PASS2_MAX_MODEL_LEN  = 8192
CONF_FLOOR           = 0.50    # raised from v5's 0.30 to cut adjacent-provision noise

print(f"QID: {QID}  TOP_K: {TOP_K}")
print(f"Output dir: {OUT_DIR}")
print("\nVerifying inputs:")
for p in [STAGE_B_IN, VAL_CSV, ALL_TARGETS, GOLD_SETS, VAL_ASPECTS]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")


## Phase 2 — Load query, aspects, candidates, gold

In [ ]:
import pandas as pd

val_df = pd.read_csv(VAL_CSV)
query_text = str(val_df[val_df.query_id == QID].iloc[0]["query"])
print(f"=== {QID} QUERY ({len(query_text)} chars) ===")
print(query_text[:1200] + ("..." if len(query_text) > 1200 else ""))

asp_df = pd.read_parquet(VAL_ASPECTS)
aspects = list(asp_df[asp_df.query_id == QID].iloc[0]["aspects"])
print(f"\n=== ASPECTS ({len(aspects)}) ===")
for a in aspects:
    print(f"  {a.get('id'):<4} (w={float(a.get('weight',0)):.2f}): {a.get('label')}")

gold_sets = json.load(open(GOLD_SETS, encoding="utf-8"))
gold_dids = set(gold_sets.get(QID, []))
total_gold = len(gold_dids)
print(f"\nGold total: {total_gold}")

sb_all = pd.read_parquet(STAGE_B_IN)
sb = (sb_all[sb_all.qid == QID].sort_values("stage_a_rank").head(TOP_K).reset_index(drop=True))
print(f"In top-{TOP_K}: {int(sb.is_gold.sum())}")

# Tier assignment
sb["tier"] = "drop"
_auto = sb.article_match & ((sb.co_citation_count >= 30) | sb.code_in_target)
sb.loc[_auto, "tier"] = "auto"
_mid = (~_auto) & (sb.article_match | (sb.co_citation_count >= 5) | (sb.concept_cosine_score >= 0.55))
sb.loc[_mid, "tier"] = "llm"
print(f"\n{sb.groupby(['tier','is_gold']).size().unstack(fill_value=0).reindex(['auto','llm','drop'], fill_value=0)}")


## Phase 3 — Pass-1 prompt (anti-hallucination)

Two new constraints over v5:
1. Explicit article-range hints (BGG has 132 articles, ZPO has 408, SchKG has 340, BV has 197).
2. Cap at 5 entries per category — forces prioritization.

In [ ]:
PASS1_PROMPT = """You are a senior Swiss attorney with deep practical knowledge of every branch of Swiss federal law (ZGB, OR, StGB, StPO, ZPO, BGG, SchKG, BV, EMRK and the BGE jurisprudence). A client has come to you with the legal question below.

CRITICAL CITATION RULES:
- Cite ONLY Swiss legal provisions you are confident actually exist. Never invent article numbers.
- Know the size of major Swiss codes when citing:
    ZGB:   articles 1-977
    OR:    articles 1-1186
    StGB:  articles 1-393
    StPO:  articles 1-457
    ZPO:   articles 1-408     (anything > 408 is NOT a real ZPO article)
    BGG:   articles 1-132     (anything > 132 is NOT a real BGG article)
    SchKG: articles 1-340
    BV:    articles 1-197
    EMRK:  articles 1-59
- If you do not know the exact article number for a rule, OMIT it rather than guess. A sparse, accurate list is far more useful than a long list with fabricated articles.
- BGE references: cite only if you are certain of the exact volume/page (e.g., "BGE 137 III 118"). Omit if uncertain.

LEGAL QUESTION FROM THE CLIENT:
{question}

THE QUESTION'S LEGAL ASPECTS (each is one sub-domain you must map):
{aspects_block}

YOUR TASK:
For EACH aspect, list the canonical Swiss legal authorities a competent Swiss lawyer would cite. Limit each category below to AT MOST 5 entries — prioritize the most foundational and well-established. Better fewer authoritative citations than a long list with weak entries.

Categories (omit empty ones):
1. FOUNDATIONAL STATUTES (the rules that directly establish the doctrine)
2. CONSTITUTIONAL/TREATY PROVISIONS (BV or EMRK, only if fundamental rights are at stake)
3. PROCEDURAL RULES (e.g., Art. 100 BGG for any Federal Tribunal appeal)
4. LEADING BGE PRECEDENTS (specific BGE references — omit if uncertain)
5. ADJACENT PROVISIONS (articles routinely co-cited in the same statutory neighborhood)

OUTPUT STRICT JSON (no prose, begin with `{{`):
{{
  "a1": {{
    "aspect_label": "<copy from input>",
    "foundational_statutes": [{{"cit": "Art. 285 ZGB", "role": "child support measurement"}}],
    "constitutional_provisions": [],
    "procedural_rules": [],
    "leading_precedents": [{{"cit": "BGE 137 III 118", "doctrine": "hypothetical income for non-earning parents"}}],
    "adjacent_provisions": [{{"cit": "Art. 286 ZGB", "role": "modification and indexing"}}]
  }},
  "a2": {{...}},
  "a3": {{...}},
  "a4": {{...}}
}}
"""

def format_aspects_block(aspects):
    parts = []
    for a in aspects:
        aid = a.get("id", "")
        lbl = a.get("label", "")
        w   = float(a.get("weight", 0))
        terms_en = ", ".join(list(a.get("concepts_en", [])))
        terms_de = ", ".join(list(a.get("terms_de", [])))
        terms_fr = ", ".join(list(a.get("terms_fr", [])))
        terms_it = ", ".join(list(a.get("terms_it", [])))
        parts.append(
            f"  Aspect {aid} (weight={w:.2f}): {lbl}\n"
            f"    English concepts: {terms_en}\n"
            f"    German terms:     {terms_de}\n"
            f"    French terms:     {terms_fr}\n"
            f"    Italian terms:    {terms_it}"
        )
    return "\n".join(parts)

pass1_prompt = PASS1_PROMPT.format(question=query_text, aspects_block=format_aspects_block(aspects))
print(f"Pass-1 prompt: {len(pass1_prompt):,} chars")


## Phase 4 — Run Pass 1

In [ ]:
from vllm import LLM, SamplingParams

print(f"Loading {LLM_MODEL} ...")
llm = LLM(
    model=LLM_MODEL, dtype="bfloat16",
    gpu_memory_utilization=0.60, max_model_len=PASS1_MAX_MODEL_LEN,
    enforce_eager=False,
)

SEEDS = [42, 43, 44]
pass1_raws = []

for seed in SEEDS:
    sp = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                        max_tokens=PASS1_MAX_TOKENS, seed=seed)
    print(f"\n--- Pass-1 sample (seed={seed}) ---")
    t0 = time.time()
    out = llm.chat([[{"role": "user", "content": pass1_prompt}]], sampling_params=sp)
    raw = out[0].outputs[0].text
    pass1_raws.append(raw)
    print(f"  Done in {time.time()-t0:.1f}s. Output: {len(raw)} chars")
print(f"\nGenerated {len(pass1_raws)} Pass-1 landscapes.")


## Phase 5 — Parse landscape + validate article ranges (anti-hallucination check)

In [ ]:
def parse_landscape(raw):
    s = raw.strip()
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL).strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]
        if s.endswith("```"):
            s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1: return None
    for cand in [s[a:b+1], s[a:b+1].replace(",}", "}").replace(",]", "]")]:
        try: return json.loads(cand)
        except Exception: pass
    return None

raw_landscapes = [parse_landscape(r) for r in pass1_raws]
ok = [l for l in raw_landscapes if l is not None]
print(f"Parsed {len(ok)} of {len(raw_landscapes)} landscapes successfully")
assert len(ok) >= 1, "All Pass-1 samples failed to parse"

# Union: per aspect, per category, dedupe by normalized citation
def norm_cit(c):
    # "Art. 285 Abs. 1 ZGB" -> "art 285 zgb" (drop Abs. lit. etc.)
    s = re.sub(r"\bAbs\.\s*\d+\w*\b", "", c or "", flags=re.IGNORECASE)
    s = re.sub(r"\blit\.\s*\w+\b", "", s, flags=re.IGNORECASE)
    s = re.sub(r"[.,]", "", s)
    return " ".join(s.lower().split())

CATEGORIES = ("foundational_statutes","constitutional_provisions",
              "procedural_rules","leading_precedents","adjacent_provisions")

merged = {}
for ls in ok:
    for aid, asp in ls.items():
        if aid not in merged:
            merged[aid] = {"aspect_label": asp.get("aspect_label",""),
                           **{c: {} for c in CATEGORIES}}
        for cat in CATEGORIES:
            items = asp.get(cat, []) or []
            for it in items:
                cit = it.get("cit","")
                if not cit: continue
                key = norm_cit(cit)
                if key not in merged[aid][cat]:
                    merged[aid][cat][key] = it

# Convert merged dicts back to lists
landscape = {}
for aid, asp in merged.items():
    landscape[aid] = {"aspect_label": asp["aspect_label"]}
    for cat in CATEGORIES:
        landscape[aid][cat] = list(asp[cat].values())

# Anti-hallucination strip (same as v5.1)
ART_LIMITS = {"ZGB": 977, "OR": 1186, "StGB": 393, "StPO": 457, "ZPO": 408,
              "BGG": 132, "SchKG": 340, "BV": 197, "EMRK": 59}
ART_RE = re.compile(r"Art\.\s*(\d+)\s*(?:Abs\.\s*\d+\w*\s*)?(?:lit\.\s*\w+\s*)?(\w+)")

def is_valid(cit):
    m = ART_RE.search(cit or "")
    if not m: return True
    num, code = int(m.group(1)), m.group(2)
    return num <= ART_LIMITS.get(code, 99999)

stripped = 0
for aid, asp in landscape.items():
    for cat in CATEGORIES:
        items = asp.get(cat, []) or []
        kept = [it for it in items if is_valid(it.get("cit",""))]
        stripped += len(items) - len(kept)
        asp[cat] = kept

print(f"\nMerged landscape (union of {len(ok)} samples); stripped {stripped} out-of-range refs")
for aid, asp in landscape.items():
    n = sum(len(asp.get(c,[])) for c in CATEGORIES)
    print(f"\n=== {aid}: {asp.get('aspect_label','')}  ({n} authorities) ===")
    for cat in CATEGORIES:
        items = asp.get(cat, [])
        if not items: continue
        print(f"  {cat}:")
        for it in items:
            cit = it.get("cit","")
            role = it.get("role","") or it.get("doctrine","") or ""
            print(f"    - {cit:<25}  {role[:90]}")


## Phase 6 — Build Pass-2 prompt (stricter)

In [ ]:
def render_landscape(landscape):
    chunks = []
    for aid in sorted(landscape.keys()):
        asp = landscape[aid]
        chunks.append(f"== {aid}: {asp.get('aspect_label','')} ==")
        for cat, hdr in [("foundational_statutes","Foundational statutes"),
                         ("constitutional_provisions","Constitutional/treaty"),
                         ("procedural_rules","Procedural rules"),
                         ("leading_precedents","Leading BGE precedents"),
                         ("adjacent_provisions","Adjacent provisions")]:
            items = asp.get(cat, [])
            if not items: continue
            chunks.append(f"  {hdr}:")
            for it in items:
                cit, role = it.get("cit",""), it.get("role","") or it.get("doctrine","")
                chunks.append(f"    - {cit:<25}  {role[:100]}")
    return "\n".join(chunks)

landscape_text = render_landscape(landscape)
print(f"Landscape: {len(landscape_text)} chars")

# Per-aspect terminology — used by Pass-2 to validate adjacent_provision claims
def aspect_terms_block(aspects):
    blocks = []
    for a in aspects:
        aid = a.get("id","")
        terms = list(a.get("terms_de",[])) + list(a.get("terms_fr",[])) + list(a.get("terms_it",[])) + list(a.get("concepts_en",[]))
        blocks.append(f"  {aid}: " + " | ".join(t for t in terms if t))
    return "\n".join(blocks)

aspect_terms_text = aspect_terms_block(aspects)

PASS2_PROMPT = """You are a senior Swiss lawyer evaluating whether to cite this candidate source. You have already mapped the canonical landscape (below). Apply the decision rules strictly — do not over-include.

LEGAL QUESTION:
{question}

CANONICAL SWISS LEGAL LANDSCAPE (your prior research):
{landscape}

PER-ASPECT KEY TERMINOLOGY (used to validate adjacent_provision claims):
{aspect_terms}

CANDIDATE SOURCE:
- Citation: {citation}
- Type: {family_label}
- {family_extra}
- Paragraph role: {role}
- Substantive text (original language):
---BEGIN---
{text}
---END---

RETRIEVAL EVIDENCE (advisory only):
- Query names this article: {article_match}  | co-citations: {co_citation_count}  | concept-cosine: {concept_cosine_score:.2f}  | code matches area: {code_in_target}

STRICT DECISION RULES (apply in order):

RULE A — Exact landscape match. Does the candidate's CITATION appear (by article number) in the foundational_statutes, constitutional_provisions, procedural_rules, or leading_precedents of any aspect in the landscape above?
  - YES -> KEEP, legal_role = matching category, confidence >= 0.85.

RULE B — Listed adjacent provision. Does the candidate's CITATION appear in the adjacent_provisions list of any aspect?
  - YES -> KEEP, legal_role = "adjacent_provision", confidence 0.65-0.80.

RULE C — Unlisted but textually on-point. The citation isn't in the landscape, BUT the text excerpt contains language that directly addresses an aspect's KEY TERMINOLOGY (the German/French/Italian terms above) AND states a rule about that exact subject (not just mentions the term).
  - YES -> KEEP, legal_role = "adjacent_provision", confidence 0.50-0.65, matched_aspect must be the specific aid.
  - This rule should fire for at most 10-20% of candidates. If you find yourself applying it to most candidates, you are being too permissive — reject instead.

RULE D — Default REJECT. If none of A, B, C apply, reject. legal_role = "off_topic", confidence <= 0.30.

OUTPUT STRICT JSON (no prose):
{{"keep": true|false, "confidence": <0.0-1.0>, "matched_aspect": "<a1|a2|a3|a4|none>", "legal_role": "<foundational_statute|constitutional|procedural_rule|leading_precedent|adjacent_provision|off_topic>", "rule_applied": "<A|B|C|D>", "reasoning": "<one sentence>"}}
"""

def family_label(r): return "Swiss court precedent paragraph" if r.family=="court" else "Swiss statutory provision"
def family_extra(r):
    if r.family=="court": return f"Court base: {r.court_base!s}. Chamber: {r.chamber!s}."
    return f"Code: {r.law_code!s}. Law: {r.law_title!s}."

def build_pass2_prompt(r):
    return PASS2_PROMPT.format(
        question=query_text[:1500],
        landscape=landscape_text,
        aspect_terms=aspect_terms_text,
        citation=r.citation,
        family_label=family_label(r),
        family_extra=family_extra(r),
        role=r.role or "(unknown)",
        article_match=str(bool(r.article_match)).lower(),
        co_citation_count=int(r.co_citation_count),
        concept_cosine_score=float(r.concept_cosine_score),
        code_in_target=str(bool(r.code_in_target)).lower(),
        text=(r.text or "")[:1200].replace('"', "'"),
    )

sb_llm = sb[sb.tier == "llm"].reset_index(drop=True)
pass2_prompts = [build_pass2_prompt(r) for r in sb_llm.itertuples()]
print(f"\nBuilt {len(pass2_prompts):,} Pass-2 prompts.")
print(f"--- Sample (first 2000 chars) ---")
print(pass2_prompts[0][:2000])


## Phase 7 — Run Pass 2

In [ ]:
sp_pass2 = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                          max_tokens=PASS2_MAX_TOKENS, seed=42)

print(f"Pass 2 on {len(pass2_prompts):,} candidates ...")
t0 = time.time()
messages = [[{"role": "user", "content": p}] for p in pass2_prompts]
outs2 = llm.chat(messages, sampling_params=sp_pass2)
raw_texts = [o.outputs[0].text for o in outs2]
print(f"Done in {(time.time()-t0)/60:.1f} min")

del llm; gc.collect()
import torch; torch.cuda.empty_cache()


## Phase 8 — Parse + composite confidence (K-cap fix)

AUTO tier confidence is now derived from `co_citation_count` (capped at 0.85), so the strongest LLM-tier keeps (conf 0.95-1.0) can outrank AUTO rows in the K-cap sort.

In [ ]:
import math

def parse_pass2(raw):
    s = raw.strip()
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL).strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]
        if s.endswith("```"):
            s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1: return None
    for cand in [s[a:b+1], s[a:b+1].replace(",}", "}").replace(",]", "]")]:
        try: return json.loads(cand)
        except Exception: pass
    return None

def auto_confidence(cc):
    """Map co_citation_count to a [0.50, 0.85] confidence so strong LLM keeps can outrank AUTO."""
    return min(0.85, 0.50 + 0.35 * math.log(max(1, cc) + 1) / 10.0)

auto_rows = [{
    "did": r.did, "is_gold": bool(r.is_gold), "tier": "auto",
    "stage_a_rank": int(r.stage_a_rank), "citation": r.citation, "family": r.family,
    "keep_llm": True,
    "confidence": auto_confidence(int(r.co_citation_count)),
    "matched_aspect": "auto", "legal_role": "auto_dossier",
    "rule_applied": "auto", "reasoning": f"strong dossier signal (cc={int(r.co_citation_count)})",
    "keep_final": True, "parse_ok": True, "raw_llm": "",
} for r in sb[sb.tier == "auto"].itertuples()]

llm_rows = []
for r, raw in zip(sb_llm.itertuples(), raw_texts):
    parsed = parse_pass2(raw) or {}
    keep_llm = bool(parsed.get("keep", False))
    conf = float(parsed.get("confidence", 0.0) or 0.0)
    matched = str(parsed.get("matched_aspect","") or "")[:8]
    legal_role = str(parsed.get("legal_role","") or "")[:60]
    rule = str(parsed.get("rule_applied","") or "")[:4]
    reasoning = str(parsed.get("reasoning","") or "")[:300]
    keep_final = keep_llm and (conf >= CONF_FLOOR)
    llm_rows.append({
        "did": r.did, "is_gold": bool(r.is_gold), "tier": "llm",
        "stage_a_rank": int(r.stage_a_rank), "citation": r.citation, "family": r.family,
        "keep_llm": keep_llm, "confidence": conf,
        "matched_aspect": matched, "legal_role": legal_role,
        "rule_applied": rule, "reasoning": reasoning,
        "keep_final": keep_final, "parse_ok": bool(parsed),
        "raw_llm": raw[:1500],
    })

out_df = pd.DataFrame(auto_rows + llm_rows)
print(f"v5.1 diagnostics:")
print(f"  AUTO kept: {(out_df.tier=='auto').sum()}  (mean conf: {out_df[out_df.tier=='auto'].confidence.mean():.2f})")
print(f"  LLM tier:  {(out_df.tier=='llm').sum()}")
sub = out_df[out_df.tier == "llm"]
if len(sub):
    print(f"    parse_ok:           {sub.parse_ok.mean():.1%}")
    print(f"    keep_llm=true:      {sub.keep_llm.mean():.1%}")
    print(f"    conf >= {CONF_FLOOR}:       {(sub.confidence >= CONF_FLOOR).mean():.1%}")
    print(f"    kept_final:         {sub.keep_final.mean():.1%}")
    print(f"  rule_applied: {sub.rule_applied.value_counts().to_dict()}")
    print(f"  legal_role:")
    print(sub.legal_role.value_counts().head(10).to_string())
print(f"  TOTAL kept: {out_df.keep_final.sum()}")


## Phase 9 — F1 (no cap + K-cap with composite confidence)

In [ ]:
def f1(p, r): return 0.0 if (p+r)==0 else 2*p*r/(p+r)

picks = set(out_df[out_df.keep_final].did)
correct = picks & gold_dids
p = len(correct) / max(1, len(picks))
r = len(correct) / max(1, total_gold)

print(f"=== v5.2 vs prior baselines for {QID} ===")
print(f"  v4:    P=0.000  R=0.000  F1=0.000  (picks=93)")
print(f"  v5:    P=0.019  R=0.500  F1=0.038  (picks=359)")
print(f"  v5.1:  P=0.024  R=0.357  F1=0.044 (no cap); K-cap F1=0.286/0.333\n  v5.2:  P={p:.3f}  R={r:.3f}  F1={f1(p,r):.3f}  (picks={len(picks)})")

# K-cap by composite confidence (now AUTO max=0.85, LLM up to 1.0)
kept = out_df[out_df.keep_final].sort_values("confidence", ascending=False)
top_K = kept.head(total_gold)
picks_k = set(top_K.did)
correct_k = picks_k & gold_dids
p_k = len(correct_k) / max(1, len(picks_k))
r_k = len(correct_k) / max(1, total_gold)
print(f"\n  K-cap (K={total_gold}, by composite conf):  P={p_k:.3f}  R={r_k:.3f}  F1={f1(p_k,r_k):.3f}")

# Also try a smaller K = gold_in_topk (10) — this is a more honest cap
gold_in_topk = int(sb.is_gold.sum())
top_K2 = kept.head(gold_in_topk)
picks_k2 = set(top_K2.did)
correct_k2 = picks_k2 & gold_dids
p_k2 = len(correct_k2)/max(1,len(picks_k2))
r_k2 = len(correct_k2)/max(1,total_gold)
print(f"  K-cap (K={gold_in_topk} = gold_in_top2000):  P={p_k2:.3f}  R={r_k2:.3f}  F1={f1(p_k2,r_k2):.3f}")

# Per-rule breakdown
print(f"\n=== Per-rule breakdown (LLM-tier kept) ===")
llm_kept = out_df[(out_df.tier=='llm') & out_df.keep_final]
for rule in sorted(llm_kept.rule_applied.unique()):
    sub = llm_kept[llm_kept.rule_applied == rule]
    n, g = len(sub), int(sub.is_gold.sum())
    print(f"  rule {rule}: {n} kept, {g} gold ({g/max(1,n):.0%})")


## Phase 10 — Per-gold verdict + reasonings

In [ ]:
gold_rows = out_df[out_df.did.isin(gold_dids)].sort_values("stage_a_rank")
print(f"=== Verdict on all 10 gold ===")
print(f"{'rank':>4}  {'cit':<25}  {'tier':<5}  {'keep':>5}  {'conf':>5}  {'rule':>4}  {'role':<22}  {'kept':>5}")
for _, r in gold_rows.iterrows():
    print(f"  {r.stage_a_rank:>4}  {r.citation[:25]:<25}  {r.tier:<5}  "
          f"{str(r.keep_llm):>5}  {r.confidence:>5.2f}  {r.rule_applied:>4}  "
          f"{r.legal_role[:22]:<22}  {str(r.keep_final):>5}")

print(f"\n=== Reasonings for kept gold ===")
for _, r in gold_rows[gold_rows.keep_final].iterrows():
    print(f"\n--- {r.citation} (rank {r.stage_a_rank}, rule {r.rule_applied}, role {r.legal_role}) ---")
    print(f"  {r.reasoning}")

print(f"\n=== Reasonings for missed gold ===")
for _, r in gold_rows[~gold_rows.keep_final].iterrows():
    print(f"\n--- {r.citation} (rank {r.stage_a_rank}, rule={r.rule_applied}, role={r.legal_role}) ---")
    print(f"  conf={r.confidence:.2f}  matched_aspect={r.matched_aspect}")
    print(f"  reasoning: {r.reasoning}")


## Phase 11 — Save outputs

In [ ]:
out_df.to_parquet(OUT_DIR / f"{QID}_v52_raw.parquet", index=False)
out_df[out_df.keep_final][["did","citation","family","tier","matched_aspect",
                            "legal_role","rule_applied","confidence","is_gold","reasoning"]].to_parquet(
    OUT_DIR / f"{QID}_v52_survivors.parquet", index=False)
with open(OUT_DIR / f"{QID}_v52_landscape.json", "w", encoding="utf-8") as fh:
    json.dump(landscape, fh, indent=2, ensure_ascii=False)
with open(OUT_DIR / f"{QID}_v52_metrics.json", "w") as fh:
    json.dump({
        "qid": QID, "config": {"TOP_K": TOP_K, "CONF_FLOOR": CONF_FLOOR},
        "tiers": {t: int((sb.tier==t).sum()) for t in ("auto","llm","drop")},
        "gold_total": total_gold, "gold_in_topk": gold_in_topk,
        "picks": len(picks), "correct": len(correct),
        "P_no_cap": p, "R_no_cap": r, "F1_no_cap": f1(p,r),
        "P_k_cap": p_k, "R_k_cap": r_k, "F1_k_cap": f1(p_k,r_k),
        "P_kgold_in_topk": p_k2, "R_kgold_in_topk": r_k2, "F1_kgold_in_topk": f1(p_k2,r_k2),
        "v4_F1": 0.000, "v5_F1": 0.038, "v51_F1_kcap": 0.286, "v51_F1_kgold": 0.333,
    }, fh, indent=2)
print(f"Saved 4 files to {OUT_DIR}")
print(f"\nFinal v5.1 F1 for {QID}: {f1(p,r):.3f} (no cap), {f1(p_k,r_k):.3f} (K-cap), {f1(p_k2,r_k2):.3f} (K=gold_in_topk)")
